# Bad Crop Evaluation

读取 ，将 score > 0.45 的图像判定为 **bad**，其余为 **good**，  
并将对应本地图像文件复制到  和  子目录，方便人工审查。

In [26]:
import os
import shutil
from pathlib import Path

# ── 配置区 ────────────────────────────────────────────────────────────────────

# bad_crop_score.txt 路径
SCORE_FILE = r"d:\Code\ms-image-quality-filters-aether-module-main\LPCreativeAdsEvaluation\bad_crop_score.txt"

# 服务器路径前缀  →  本地路径前缀
SERVER_PREFIX = "/vc_data/shares/bingads.algo.prod.adsplus/ProdAdsPlusShare/Team/RichAds/AIGC/CKPT/ZImage/Official"
LOCAL_PREFIX  = r"D:\Data\T2I\ZImage\official"

# bad crop 判定阈值
BAD_THRESHOLD = 0.05

# 输出根目录（bad/ 和 good/ 将创建在此目录下）
OUTPUT_ROOT = Path(r"D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636")
BAD_DIR  = OUTPUT_ROOT / "bad"
GOOD_DIR = OUTPUT_ROOT / "good"

# ─────────────────────────────────────────────────────────────────────────────

BAD_DIR.mkdir(exist_ok=True)
GOOD_DIR.mkdir(exist_ok=True)

print(f"Score 文件  : {SCORE_FILE}")
print(f"Bad  阈值   : score > {BAD_THRESHOLD}")
print(f"Bad  输出目录: {BAD_DIR}")
print(f"Good 输出目录: {GOOD_DIR}")

Score 文件  : d:\Code\ms-image-quality-filters-aether-module-main\LPCreativeAdsEvaluation\bad_crop_score.txt
Bad  阈值   : score > 0.05
Bad  输出目录: D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\bad
Good 输出目录: D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\good


In [27]:
def server_to_local(server_path: str) -> Path:
    """将服务器路径转换为本地 Windows 路径。"""
    rel = server_path[len(SERVER_PREFIX):].lstrip("/")
    return Path(LOCAL_PREFIX) / Path(rel.replace("/", os.sep))


def parse_score_file(score_file: str):
    """逐行解析 score 文件，返回 (server_path, score) 列表。"""
    records = []
    with open(score_file, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = line.rsplit("	", 1)   # 用最后一个 Tab 分割
            if len(parts) != 2:
                print(f"[警告] 第 {lineno} 行格式异常，已跳过: {line[:80]}")
                continue
            path_str, score_str = parts
            try:
                score = float(score_str)
            except ValueError:
                print(f"[警告] 第 {lineno} 行 score 无法解析，已跳过: {score_str}")
                continue
            records.append((path_str.strip(), score))
    return records


records = parse_score_file(SCORE_FILE)
print(f"共读取到 {len(records)} 条记录")

共读取到 1500 条记录


In [28]:
bad_count  = 0
good_count = 0
miss_count = 0
miss_files = []

for server_path, score in records:
    local_path = server_to_local(server_path)
    label      = "bad" if score > BAD_THRESHOLD else "good"
    dest_dir   = BAD_DIR if label == "bad" else GOOD_DIR

    if not local_path.exists():
        miss_count += 1
        miss_files.append(str(local_path))
        continue

    # 保留原文件名，若目标已有同名文件则覆盖
    shutil.copy2(local_path, dest_dir / local_path.name)

    if label == "bad":
        bad_count += 1
    else:
        good_count += 1

print(f"{chr(61)*50}")
print(f"处理完成")
print(f"  Bad  (score > {BAD_THRESHOLD}): {bad_count}  张  →  {BAD_DIR}")
print(f"  Good (score <= {BAD_THRESHOLD}): {good_count}  张  →  {GOOD_DIR}")
print(f"  本地文件不存在  : {miss_count}  张")
if miss_files:
    print("缺失文件列表（前 20 条）:")
    for p in miss_files[:20]:
        print(f"  {p}")

处理完成
  Bad  (score > 0.05): 126  张  →  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\bad
  Good (score <= 0.05): 552  张  →  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\good
  本地文件不存在  : 822  张
缺失文件列表（前 20 条）:
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\Internal10050_Prompt2_1280x720.png
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\Internal10050_Prompt2_1536x864.png
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\Internal10050_Prompt2_2048x1152.png
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\Internal10050_Prompt3_1280x720.png
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_20260305-0636\Internal10050_Prompt3_1536x864.png
  D:\Data\T2I\ZImage\official\LP100_0131-2_16x9_official_ZImage_images_cfg0_202

In [6]:
# ── 可选：以 DataFrame 形式查看 bad / good 分类结果 ──────────────────────────
import pandas as pd

df = pd.DataFrame(records, columns=["server_path", "score"])
df["filename"] = df["server_path"].apply(lambda p: Path(p).name)
df["label"]    = df["score"].apply(lambda s: "bad" if s > BAD_THRESHOLD else "good")

df_bad  = df[df["label"] == "bad"].sort_values("score", ascending=False).reset_index(drop=True)
df_good = df[df["label"] == "good"].sort_values("score", ascending=False).reset_index(drop=True)

print(f"Bad 图像（共 {len(df_bad)} 张）：")
display(df_bad[["filename", "score"]])

print(f"Good 图像（共 {len(df_good)} 张）：")
display(df_good[["filename", "score"]])

Bad 图像（共 58 张）：


,filename,score
0,Internal10039_Prompt5_1280x720.png,0.731934
1,Internal10049_Prompt1_2048x1152.png,0.706055
2,Internal10051_Prompt5_1536x864.png,0.702148
3,Internal10083_Prompt3_2048x1152.png,0.689453
4,Internal10069_Prompt4_2048x1152.png,0.688965
5,Internal10051_Prompt5_1280x720.png,0.681152
6,Internal10021_Prompt3_2048x1152.png,0.667969
7,Internal10051_Prompt5_2048x1152.png,0.652344
8,Internal10051_Prompt4_1280x720.png,0.642578
9,Internal10083_Prompt3_1536x864.png,0.636230


Good 图像（共 1442 张）：


,filename,score
0,Internal10036_Prompt5_2048x1152.png,0.447510
1,Internal10040_Prompt1_2048x1152.png,0.444824
2,Internal1005_Prompt4_1536x864.png,0.444824
3,Internal10075_Prompt4_2048x1152.png,0.443604
4,Internal10072_Prompt4_2048x1152.png,0.441406
...,...,...
1437,Internal10018_Prompt3_1536x864.png,0.018295
1438,Internal10048_Prompt5_2048x1152.png,0.018295
1439,Internal10045_Prompt5_1280x720.png,0.018158
1440,Internal1008_Prompt4_1280x720.png,0.017990
